In [26]:
spark.sql("select * from spark_db.flight_time_raw").show()

+----------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|CRS_DEP_TIME|DEP_TIME|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|CANCELLED|DISTANCE|
+----------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1115|    1113|     1343|      5|        1400|    1348|        0|     946|
|2000-01-01|        DL|             1479|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1315|    1311|     1536|      7|        1559|    1543|        0|     946|
|2000-01-01|        DL|             1857|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1415|    1414|     1642|      9|        1721|    1651|        0|     946

In [29]:
"""
Apply transformations to time values as hour to minute interval

CRS_DEP_TIME
DEP_TIME
WHEELS_ON
CRS_ARR_TIME
ARR_TIME
"""
flight_time_raw_df = spark.read.table("spark_db.flight_time_raw")

In [41]:
from pyspark.sql.functions import expr

step_1_df = (
    flight_time_raw_df.withColumns({
        "CRS_DEP_TIME_HH": expr("left(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
        "CRS_DEP_TIME_MM": expr("right(lpad(CRS_DEP_TIME, 4, '0'), 2)")
    })
)

step_2_df = (
    step_1_df.withColumns({
        "CRS_DEP_TIME_NEW": expr("cast(CRS_DEP_TIME_HH || ':' || CRS_DEP_TIME_MM as INTERVAL HOUR TO MINUTE)")
    })
)
step_2_df.show()

+----------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+---------------+---------------+--------------------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|CRS_DEP_TIME|DEP_TIME|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|CANCELLED|DISTANCE|CRS_DEP_TIME_HH|CRS_DEP_TIME_MM|    CRS_DEP_TIME_NEW|
+----------+----------+-----------------+------+----------------+----+--------------+------------+--------+---------+-------+------------+--------+---------+--------+---------------+---------------+--------------------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1115|    1113|     1343|      5|        1400|    1348|        0|     946|             11|             15|INTERVAL '11:15' ...|
|2000-01-01|        DL|             1479|   BOS|      Boston, MA| ATL|   Atlanta, GA|        1315|    1311|     1536|   

In [42]:
# Build a reuable function to modify all the columns as needed

def get_interval(hhmm_value):
    from pyspark.sql.functions import expr

    return expr(
        f"""
        cast(left(lpad({hhmm_value}, 4, '0'), 2) || ':' || right(lpad({hhmm_value}, 4, '0'), 2) as INTERVAL HOUR TO MINUTE)
        """
    )

In [49]:
"""
Apply transformation to TAXI_IN to make it a minute interval (EXTRA)
"""

result_df = (
    flight_time_raw_df.withColumns({
        "CRS_DEP_TIME": get_interval("CRS_DEP_TIME"),
        "DEP_TIME": get_interval("DEP_TIME"),
        "WHEELS_ON": get_interval("WHEELS_ON"),
        "CRS_ARR_TIME": get_interval("CRS_ARR_TIME"),
        "ARR_TIME": get_interval("ARR_TIME"),
        "TAX_IN": expr("cast(TAXI_IN AS INTERVAL MINUTE)") # no need function for this as TAXI_IN always had to be minutes
    })
)
result_df.show()

+----------+----------+-----------------+------+----------------+----+--------------+--------------------+--------------------+--------------------+-------+--------------------+--------------------+---------+--------+--------------------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|        CRS_DEP_TIME|            DEP_TIME|           WHEELS_ON|TAXI_IN|        CRS_ARR_TIME|            ARR_TIME|CANCELLED|DISTANCE|              TAX_IN|
+----------+----------+-----------------+------+----------------+----+--------------+--------------------+--------------------+--------------------+-------+--------------------+--------------------+---------+--------+--------------------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|INTERVAL '11:15' ...|INTERVAL '11:13' ...|INTERVAL '13:43' ...|      5|INTERVAL '14:00' ...|INTERVAL '13:48' ...|        0|     946|INTERVAL '05' MINUTE|
|2000-01-01|        DL|             1479|   

In [50]:
result_df.write.mode("spark_db.flight_time_clean")

AttributeError: 'DataFrameWriter' object has no attribute 'table'